In [1]:
import cloudscraper
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
from functools import reduce
from time import sleep
import random

HEADERS_LIST = [
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    },
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_5) "
                      "AppleWebKit/605.1.15 (KHTML, like Gecko) "
                      "Version/16.4 Safari/605.1.15",
        "Accept-Language": "en-US,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    },
    {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:116.0) "
                      "Gecko/20100101 Firefox/116.0",
        "Accept-Language": "en-GB,en;q=0.7",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    },
    {
        "User-Agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 16_4 like Mac OS X) "
                      "AppleWebKit/605.1.15 (KHTML, like Gecko) "
                      "CriOS/115.0.5790.110 Mobile/15E148 Safari/604.1",
        "Accept-Language": "en-US,en;q=0.6",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    },
    {
        "User-Agent": "Mozilla/5.0 (Linux; Android 14; Pixel 7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/115.0.5790.170 Mobile Safari/537.36",
        "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    },
]


def fetch_with_random_header(url: str, **kwargs):
    """
    Realiza una petición GET con un header aleatorio de HEADERS_LIST.
    """
    COMMON_HEADERS = {
            **random.choice(HEADERS_LIST),
            "Referer": "https://fbref.com/",
            "Connection": "keep-alive",
        } 
    return scraper.get(url, headers=COMMON_HEADERS, timeout=30)
    

def get_team_links(league:str, league_url: str) -> list[tuple[str, str]]:

    resp = fetch_with_random_header(league_url, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    table = soup.find("div", {"id":"all_results2024-202591"}).find("table", {"id":"results2024-202591_overall"})
    if table is None or table.tbody is None:
        raise ValueError("No se encontró la tabla de squads. ¿Ha cambiado el ID?")

    links: list[tuple[str, str]] = []
    for row in table.tbody.find_all("tr"):
        cell = row.find("td", {"class":"left","data-stat": "team"})
        if cell is None:
            continue
        a = cell.find("a", href=True)
        if not a:
            continue
        team_name = a.text.strip()
        team_href = urljoin("https://fbref.com", a["href"])
        links.append((team_name, team_href, league))

    return links

    
def scrape_team_stats(team_name: str, team_url: str, league_name: str) -> pd.DataFrame:
    """
    Scrapea todas las tablas de estadísticas de jugadores de un equipo,
    renombra columnas, añade logo y devuelve un único DataFrame con Team y League.
    """
    resp = fetch_with_random_header(team_url, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Team Logo
    logo_tag = soup.find("img", {"class": "teamlogo"})
    team_logo = urljoin(team_url, logo_tag.get("src")) if logo_tag and logo_tag.get("src") else None

    table_ids = [
    "stats_standard_9",
    "stats_keeper_adv_9",
    "stats_shooting_9",
    "stats_passing_9",
    "stats_passing_types_9",
    "stats_gca_9",
    "stats_defense_9",
    "stats_possession_9",
    "stats_playing_time_9",
]

    dfs = []
    for i,tid in enumerate(table_ids):
        try:
            table = soup.find("table", {"id": tid})
            
            header_rows = table.thead.find_all("tr")
           
            cols = [th["data-stat"] for th in header_rows[1].find_all("th")]
            
            records = []
            for tr in table.tbody.find_all("tr"):
                cells = tr.find_all(["th", "td"])
                if len(cells) != len(cols):
                # Avoid irregular rows as <tfoot>
                    continue
            
                row = {}
                for col, cell in zip(cols, cells):
                # Si hay un <a> dentro de la celda, tomar su texto
                    a = cell.find("a")
                    text = a.text.strip() if a else cell.get_text(strip=True)
                    row[col] = text
                records.append(row)
            
            # Creating DataFrame
            df = pd.DataFrame.from_records(records)
            df["nationality"] = df["nationality"].apply(lambda x: x.split(" ")[1])
            if "minutes_90s" in df.columns and i !=0:
                df.drop("minutes_90s", axis=1, inplace=True)
            dfs.append(df)
        except:
            print(f"Failed retreiving {tid}")

    if not dfs:
        return pd.DataFrame()

    df_all = reduce(lambda l, r: pd.merge(l, r, on=["player", "nationality", "position","age", "matches"], how="left"), dfs)
    df_all["Team"] = team_name
    df_all["League"] = league_name
    df_all["Team_Logo"] = team_logo

    bad_cols = []
    for c in df_all.columns:
        if "_y" == c[-2:]:
            bad_cols.append(c)
    df_all = df_all.drop(bad_cols, axis=1)

    cols = [c.replace("_x", "") for c in df_all.columns]

    df_all.columns = cols
    
    return df_all

In [3]:
df1 = pd.read_csv("./data/players_all_leagues_until_ Belgian Pro League.csv")
df2 = pd.read_csv("./data/players_all_leagues_until_Saudi Pro League.csv")
df3 = pd.read_csv("./data/players_all_leagues_until_Serie B.csv")
df4 = pd.read_csv("./data/players_all_leagues_until_Veikkausliiga.csv")

In [4]:
df_all = pd.concat([df1, df2], ignore_index=True, sort=False)

In [6]:
df_all = pd.concat([df_all, df3, df4], ignore_index=True, sort=False)

In [13]:
df_all["nationalit"] = df_all["nationalit"].fillna("OTH")

In [15]:
df_all["position"] = df_all["position"].fillna("SUB")

In [17]:
df_all = df_all.fillna(0)

In [22]:
df_all["age"] = df_all["age"].apply(lambda x: int(x) if "-" not in str(x) else int(str(x).split("-")[0])) 

In [25]:
df_all["position"] = df_all["position"].apply(lambda x: x if "," not in x else x.split(",")[0])

In [27]:
df_all.rename(columns={"nationalit": "nationality"}, inplace=True)

In [29]:
df_all.drop("matches", axis=1, inplace=True)

In [30]:
df_all.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18590 entries, 0 to 18589
Data columns (total 191 columns):
 #    Column                                 Non-Null Count  Dtype  
---   ------                                 --------------  -----  
 0    player                                 18590 non-null  object 
 1    nationality                            18590 non-null  object 
 2    position                               18590 non-null  object 
 3    age                                    18590 non-null  int64  
 4    games                                  18590 non-null  int64  
 5    games_starts                           18590 non-null  int64  
 6    minutes                                18590 non-null  object 
 7    minutes_90s                            18590 non-null  float64
 8    goals                                  18590 non-null  float64
 9    assists                                18590 non-null  float64
 10   goals_assists                          18590 non-null  f

In [31]:
df_all.to_csv("data/all_players_cleaned.csv", index=False)

In [23]:
urls = [
    #["https://fbref.com/en/comps/12/La-Liga-Stats","La Liga"],
    #["https://fbref.com/en/comps/9/Premier-League-Stats","Premier League"],
    #["https://fbref.com/en/comps/11/Serie-A-Stats","Serie A"],
    #["https://fbref.com/en/comps/20/Bundesliga-Stats","Bundesliga"],
    #["https://fbref.com/en/comps/13/Ligue-1-Stats","Ligue 1"],
    #["https://fbref.com/en/comps/22/Major-League-Soccer-Stats","Major League Soccer"],
    #["https://fbref.com/en/comps/10/Championship-Stats","Championship"],
    #["https://fbref.com/en/comps/24/Serie-A-Stats","Brasileirao"],
    #["https://fbref.com/en/comps/32/Primeira-Liga-Stats","Primeira Liga"],
    #["https://fbref.com/en/comps/23/Eredivisie-Stats","Eredivisie"],
    #["https://fbref.com/en/comps/31/Liga-MX-Stats","Liga MX"],
    #["https://fbref.com/en/comps/21/Liga-Profesional-Argentina-Stats","Liga Argentina"],
    #["https://fbref.com/en/comps/37/Belgian-Pro-League-Stats"," Belgian Pro League"],
    ["https://fbref.com/en/comps/17/Segunda-Division-Stats","Liga Hipermotion"],
    ["https://fbref.com/en/comps/25/J1-League-Stats","J1 League"],
    ["https://fbref.com/en/comps/26/Super-Lig-Stats","Sper Lig"],
    ["https://fbref.com/en/comps/38/Serie-B-Stats","Brasileirao B"],
    ["https://fbref.com/en/comps/70/Saudi-Professional-League-Stats","Saudi Pro League"],
    ["https://fbref.com/en/comps/18/Serie-B-Stats","Serie B"],
    ["https://fbref.com/en/comps/56/history/Austrian-Bundesliga-Seasons","Austrian Bundesliga"],
    ["https://fbref.com/en/comps/67/history/Bulgarian-First-League-Seasons","Bulgarian First League"],
    ["https://fbref.com/en/comps/211/history/Canadian-Premier-League-Seasons","Canadian Premier League"],
    ["https://fbref.com/en/comps/62/history/Chinese-Super-League-Seasons","Chinese Super League"],
    ["https://fbref.com/en/comps/63/history/Hrvatska-NL-Seasons","Croatian League"],
    ["https://fbref.com/en/comps/50/history/Danish-Superliga-Seasons","Danish Superliga"],
    ["https://fbref.com/en/comps/27/history/Super-League-Greece-Seasons","Greece Super League"],
    ["https://fbref.com/en/comps/55/history/K-League-1-Seasons","Korean League 1"],
    ["https://fbref.com/en/comps/47/history/Liga-I-Seasons","Roumanian League I"],
    ["https://fbref.com/en/comps/57/history/Swiss-Super-League-Seasons","Swiss Super League"],
    ["https://fbref.com/en/comps/28/history/Eliteserien-Seasons","Eliteserien"],
    ["https://fbref.com/en/comps/66/history/Czech-First-League-Seasons","Czech First League"],
    ["https://fbref.com/en/comps/43/history/Veikkausliiga-Seasons","Veikkausliiga"]
]

In [24]:
all_data = []

In [25]:
# Creating scraper
scraper = cloudscraper.create_scraper()

for url in urls:
    print(f"Getting data from {url[1]}")
    links = get_team_links(url[1],url[0])
    for link in links:
        print(f"Getting data from {link[0]}")
        team = scrape_team_stats(link[0], link[1], link[2])
        all_data.append(team)
        sleep(10)

Getting data from La Liga


HTTPError: 429 Client Error: Too Many Requests for url: https://fbref.com/en/comps/12/La-Liga-Stats